# 🗓️ Generador de Líneas de Tiempo — MIMP

**Solo haz clic en ▶ y espera. Después sube tu archivo.**

---

In [ ]:
# @title ▶ Haz clic aquí para iniciar
import subprocess, sys, shutil, zipfile, urllib.request
from pathlib import Path
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

display(HTML("<div style='font-family:Arial;font-size:16px;padding:12px;background:#fff3cd;border-radius:8px'>⏳ Preparando el sistema... (~30 segundos)</div>"))

# Instalar dependencias
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "openpyxl","python-pptx","python-docx","pdfplumber",
    "dateparser","google-genai","Pillow","lxml","ipywidgets"],
    check=True, capture_output=True)

# Limpiar versión anterior
proj_dir = Path("/content/timeline_generator")
if proj_dir.exists(): shutil.rmtree(proj_dir)
for k in [k for k in sys.modules if k.startswith(("core","utils"))]: del sys.modules[k]

# Descargar código desde GitHub
ZIP_URL = "https://github.com/KEVINmarce1996/timeline-generator-mimp/raw/main/timeline_generator_v2%20%2824%29.zip"
zip_path = Path("/content/tg.zip")
urllib.request.urlretrieve(ZIP_URL, zip_path)
with zipfile.ZipFile(zip_path) as z:
    z.extractall("/content")
sys.path.insert(0, "/content/timeline_generator")

# Cargar módulos
import importlib, logging
logging.basicConfig(level=logging.WARNING)
import core.model_analyzer;  importlib.reload(core.model_analyzer)
import core.file_parser;      importlib.reload(core.file_parser)
import core.timeline_builder; importlib.reload(core.timeline_builder)
import core.pptx_generator;   importlib.reload(core.pptx_generator)
import utils.date_detector;   importlib.reload(utils.date_detector)
from core.model_analyzer   import ModelAnalyzer
from core.file_parser      import FileParser
from core.timeline_builder import TimelineBuilder, BuilderConfig
from core.pptx_generator   import PptxGenerator
from utils.date_detector   import DetectorConfig
from datetime import date

template = ModelAnalyzer(
    "/content/timeline_generator/assets/formato_excel_modelo.xlsx").extract()

# ── Mostrar interfaz ─────────────────────────────────────────────────────────
clear_output(wait=True)

display(HTML("""
<div style="font-family:Arial;max-width:580px;margin:0 auto">
  <div style="background:#1F497D;color:white;padding:18px 22px;border-radius:10px 10px 0 0">
    <h2 style="margin:0;font-size:20px">🗓️ Generador de Líneas de Tiempo</h2>
    <p style="margin:4px 0 0;opacity:.85;font-size:13px">MIMP — Proyectos de Inversión Pública</p>
  </div>
  <div style="background:#f8f9fa;padding:18px 22px;border:1px solid #dee2e6;
              border-radius:0 0 10px 10px">
    <p style="margin:0 0 6px;font-size:14px;color:#444">
      📎 <strong>Sube tu archivo</strong> con el cronograma del proyecto
    </p>
    <p style="margin:0;font-size:12px;color:#888">
      ✅ Imagen (PNG/JPG) &nbsp;|&nbsp; Word (DOCX) &nbsp;|&nbsp; PDF &nbsp;|&nbsp; Excel (XLSX)
    </p>
  </div>
</div>
"""))

upload_btn = widgets.FileUpload(
    accept=".png,.jpg,.jpeg,.docx,.pdf,.xlsx,.xls",
    multiple=False,
    description="📂 Subir archivo",
    button_style="primary",
    layout=widgets.Layout(width="200px", margin="12px auto 0")
)
status_out = widgets.Output()

def procesar(change):
    if not upload_btn.value: return
    with status_out:
        clear_output(wait=True)
        fname   = list(upload_btn.value.keys())[0]
        content = list(upload_btn.value.values())[0]["content"]
        display(HTML(f"""
        <div style="font-family:Arial;max-width:580px;margin:8px auto;
                    background:#fff3cd;border:1px solid #ffc107;
                    border-radius:8px;padding:12px 16px;font-size:13px">
          ⏳ Procesando <strong>{fname}</strong>...
        </div>"""))
        try:
            udir = Path("/content/mis_archivos"); udir.mkdir(exist_ok=True)
            dest = udir / fname
            dest.write_bytes(bytes(content))
            parser  = FileParser(DetectorConfig(granularity="annual"), use_vision=True)
            builder = TimelineBuilder(BuilderConfig(granularity="annual", max_columns=14))
            parsed  = parser.parse(dest)
            layout  = builder.build(parsed)
            out_dir = Path("/content/output"); out_dir.mkdir(exist_ok=True)
            pptx_path = out_dir / "timeline.pptx"
            gen = PptxGenerator(template)
            gen.add_layout(layout)
            gen.save(pptx_path)
            # Tabla de hitos
            rows = ""
            for s in layout.sections:
                for col in s.columns:
                    for e in col.events:
                        d = e.detected_date.date_start.strftime("%d/%m/%Y") \
                            if e.detected_date and e.detected_date.date_start else "-"
                        bg  = {"done":"#f8f9fa","active":"#cfe2ff","pending":"#fff3cd"}.get(e.status,"#fff")
                        est = {"done":"✅ Ejecutado","active":"🔵 En curso","pending":"🟡 Pendiente"}.get(e.status,"")
                        rows += (f'<tr style="background:{bg}">'
                                 f'<td style="padding:5px 10px;font-size:12px">{est}</td>'
                                 f'<td style="padding:5px 10px;font-size:12px">{d}</td>'
                                 f'<td style="padding:5px 10px;font-size:12px">{e.label[:50]}</td></tr>')
            display(HTML(f"""
            <div style="font-family:Arial;max-width:580px;margin:8px auto">
              <div style="background:#d1e7dd;border:1px solid #a3cfbb;border-radius:8px;
                          padding:12px 16px;margin-bottom:10px;font-size:13px">
                ✅ <strong>¡Línea de tiempo lista!</strong> — {layout.project_title}
              </div>
              <table style="width:100%;border-collapse:collapse;font-size:12px;
                            border:1px solid #dee2e6;border-radius:6px;overflow:hidden">
                <tr style="background:#1F497D;color:white">
                  <th style="padding:6px 10px;text-align:left">Estado</th>
                  <th style="padding:6px 10px;text-align:left">Fecha</th>
                  <th style="padding:6px 10px;text-align:left">Etapa</th></tr>
                {rows}
              </table>
            </div>"""))
            from google.colab import files as cf
            cf.download(str(pptx_path))
            display(HTML("""
            <div style="font-family:Arial;max-width:580px;margin:8px auto;
                        background:#cfe2ff;border:1px solid #9ec5fe;border-radius:8px;
                        padding:12px 16px;text-align:center;font-size:13px">
              📥 <strong>PowerPoint descargándose</strong><br>
              <span style="color:#555">Revisa tu carpeta de Descargas</span>
            </div>"""))
        except Exception:
            import traceback
            tb = traceback.format_exc()
            display(HTML(f"""
            <div style="font-family:Arial;max-width:580px;margin:8px auto;
                        background:#f8d7da;border:1px solid #f5c2c7;border-radius:8px;
                        padding:12px 16px;font-size:12px">
              ❌ <strong>Error procesando el archivo</strong><br>
              <pre style="white-space:pre-wrap;font-size:10px">{tb[:600]}</pre>
            </div>"""))

upload_btn.observe(procesar, names="value")
display(widgets.VBox([upload_btn, status_out],
    layout=widgets.Layout(align_items="center")))
